In [3]:
# ── Cell 0: Setup (chạy đầu tiên) ──
import os

# Clone repo nếu chưa có
if not os.path.exists('/content/project'):
    !git clone https://github.com/TruongDuyLongPTIT/CTQW_PRO_METABOLITES_PRIORITIZING.git /content/project

# Add src vào Python path
import sys
sys.path.insert(0, '/content/project/src')

# Install dependencies nếu cần
!pip install -q torch scikit-learn networkx tqdm

In [4]:
from pathlib import Path
from google.colab import drive

if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive')
else:
    print("Drive đã mount sẵn.")

Mounted at /content/drive


In [ ]:
!python /content/project/experiments/01_main_results.py

STEP 1 — Build graph
  Graph: 2788 nodes, 22439 edges
  G_pro: 2894 nodes (106 pathway), 31360 edges

STEP 2 — Build eval sets
  hmdb_to_recon: +0 IK, +331 name → 3286 total
  Extracting SMPDB metabolites...
  Disease pathways: 20248
SMPDB files: 100% 48687/48687 [00:38<00:00, 1252.91it/s]
  Loaded 20244 pathways (errors=0)
  Dedup: 20244 → 3240 → 362 → 215
  eval_set1 (HMDB+CTD): 158 diseases
  eval_set2 (MarkerDB): 21 diseases
  eval_set3 (SMPDB):    153 diseases

STEP 3 — Eigendecomposition
  Done.

[Table 1] RWR vs CTQW on G_cc...
RWR/HMDB+CTD: 100% 158/158 [11:04<00:00,  4.20s/it]
CTQW/HMDB+CTD: 100% 158/158 [07:41<00:00,  2.92s/it]
  HMDB+CTD: 18.8 min
RWR/MarkerDB: 100% 21/21 [02:00<00:00,  5.73s/it]
CTQW/MarkerDB: 100% 21/21 [01:21<00:00,  3.89s/it]
  MarkerDB: 3.4 min
RWR/SMPDB: 100% 153/153 [08:29<00:00,  3.33s/it]
CTQW/SMPDB: 100% 153/153 [05:33<00:00,  2.18s/it]
  SMPDB: 14.0 min

[Table 2] PROFANCY vs CTQW-PRO on G_pro...
PROFANCY/HMDB+CTD: 100% 158/158 [09:37<00:00,  3.65

In [ ]:
!python /content/project/experiments/02_ablation_graph.py

Building graphs...
  G_pro: 2894 → clean: 2851 nodes
Building eval sets...
  Disease pathways: 20248
SMPDB files: 100% 48687/48687 [00:42<00:00, 1152.44it/s]
  Loaded 20244 pathways (errors=0)
  Dedup: 20244 → 3240 → 362 → 215
Eigendecomposition...

Running ablation...
PROF_o/HMDB+CTD: 100% 158/158 [09:20<00:00,  3.55s/it]
CTQW_o/HMDB+CTD: 100% 158/158 [10:24<00:00,  3.96s/it]
PROF_c/HMDB+CTD: 100% 158/158 [13:32<00:00,  5.14s/it]
CTQW_c/HMDB+CTD: 100% 158/158 [09:12<00:00,  3.50s/it]
  HMDB+CTD: 42.5 min
PROF_o/MarkerDB: 100% 21/21 [01:41<00:00,  4.83s/it]
CTQW_o/MarkerDB: 100% 21/21 [01:53<00:00,  5.39s/it]
PROF_c/MarkerDB: 100% 21/21 [02:24<00:00,  6.89s/it]
CTQW_c/MarkerDB: 100% 21/21 [01:40<00:00,  4.77s/it]
  MarkerDB: 7.7 min
PROF_o/SMPDB: 100% 153/153 [07:07<00:00,  2.79s/it]
CTQW_o/SMPDB: 100% 153/153 [07:51<00:00,  3.08s/it]
PROF_c/SMPDB: 100% 153/153 [10:13<00:00,  4.01s/it]
CTQW_c/SMPDB: 100% 153/153 [06:54<00:00,  2.71s/it]
  SMPDB: 32.1 min

ABLATION: Original vs Clean G_

In [ ]:
!python /content/project/experiments/03_negative_results.py

Setup...
  Disease pathways: 20248
SMPDB files: 100% 48687/48687 [00:40<00:00, 1207.92it/s]
  Loaded 20244 pathways (errors=0)
  Dedup: 20244 → 3240 → 362 → 215
  HMDB+CTD: 158 diseases
  SMPDB:    153 diseases

EXP 1: Self-loop leakage analysis
  (A) Baseline:     CTQW-PRO, γ=0
  (B) Leaked:       H = A_pro + γ·diag(ALL mets)
  (C) Leakage-free: H = A_pro + γ·diag(seeds_only) per fold
Traceback (most recent call last):
  File "/content/project/experiments/03_negative_results.py", line 135, in <module>
    ev_c, vecs_c = np.linalg.eigh(H_c); del H_c
                   ^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/numpy/linalg/_linalg.py", line 1601, in eigh
    w, vt = gufunc(a, signature=signature)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt
^C


In [5]:
# ═══════════════════════════════════════════════════════════════════
# TEST: Quantum Barrier (On-site Potential) tại cofactor nodes
# 3 options: uniform, degree-proportional, normalized-degree
# Copy toàn bộ vào 1 cell Colab và chạy (~20-30 phút)
# ═══════════════════════════════════════════════════════════════════
import sys, os
for _p in ['/content/project/src',
           '/content/drive/MyDrive/CTQW for metabolites/src']:
    if os.path.isdir(_p) and _p not in sys.path:
        sys.path.insert(0, _p)

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from config import CACHE_DIR, RESULTS_DIR, T_FIXED, NH_GAMMA, RECON3D_COFACTORS
from graph import (parse_recon3d, build_gcc, build_gpro,
                   build_hmdb_to_recon_initial, augment_hmdb_to_recon,
                   compute_eigendecomp)
from eval_sets import (parse_hmdb, build_hmdb_lookups,
                       build_cofactors_set, build_eval_set3)
from methods import make_nh_pro
from evaluation import compute_metrics

# ── Params ────────────────────────────────────────────────────────
N_DISEASES = 30   # sample nhanh; đổi None để full (~153)

# ── Setup ─────────────────────────────────────────────────────────
print('Setup...')
recon_data   = parse_recon3d()
G_cc, graph_nodes, N, node_idx, A_cc, degrees = build_gcc(recon_data)
met_info     = recon_data['met_info']
pathway_mets = recon_data['pathway_mets']

(G_pro, pro_nodes, N_PRO, idx_pro,
 A_pro, deg_pro, _pro_src, _pro_dst) = build_gpro(G_cc, node_idx, pathway_mets)

hmdb_data        = parse_hmdb()
hmdb_metabolites = hmdb_data['metabolites']
hmdb_lookups     = build_hmdb_lookups(hmdb_metabolites)
hmdb_to_recon    = build_hmdb_to_recon_initial(met_info, node_idx)
augment_hmdb_to_recon(
    hmdb_to_recon, met_info, node_idx,
    hmdb_lookups['ik_to_id'], hmdb_lookups['ikshort_to_id'],
    hmdb_lookups['name_to_id'], hmdb_lookups['name_aggr_to_id'])
COFACTORS = build_cofactors_set(hmdb_metabolites)
eval_set3 = build_eval_set3(hmdb_metabolites, hmdb_to_recon, node_idx)
print(f'  SMPDB: {len(eval_set3)} diseases')

# ── Eigendecomp G_pro (dùng cho CTQW baseline và QB methods) ──────
eigvals_pro, eigvecs_pro = compute_eigendecomp(A_pro, CACHE_DIR / 'gpro_eigdecomp.npz')

# ── Xác định cofactor nodes và degree trong G_pro ─────────────────
cof_set  = set(RECON3D_COFACTORS)
cof_info = {}   # {pro_idx: degree}
for i, nd in enumerate(pro_nodes):
    nd_base = nd.replace('_c','').replace('_m','').replace('_e','').replace('_x','')
    if nd in cof_set or nd_base in cof_set:
        cof_info[i] = deg_pro[i]

cof_degrees   = np.array(list(cof_info.values()))
cof_mean_deg  = float(cof_degrees.mean())
graph_mean_deg = float(deg_pro.mean())  # = NH_GAMMA ≈ 22

print(f'  Cofactor nodes: {len(cof_info)}')
print(f'  Cofactor mean degree: {cof_mean_deg:.1f}')
print(f'  Graph mean degree (NH_GAMMA): {graph_mean_deg:.1f}')

# ── Builder: quantum barrier runner ──────────────────────────────
def make_quantum_barrier(potential_dict, label='QB'):
    """
    H_eff = A_pro + diag(v)
    potential_dict: {pro_node_idx: v_i}  (real values, Hermitian)

    Khác với NH-CTQW:
      NH:  H_eff[i,i] -= 1j*gamma  → imaginary → amplitude decay
      QB:  H_eff[i,i] += v_i       → real      → phase shift (tunneling analog)
    """
    H_eff = A_pro.astype(complex)
    for i, v in potential_dict.items():
        H_eff[i, i] += float(v)   # REAL: Hermitian Hamiltonian

    # Dùng eigh vì H_eff vẫn Hermitian (real symmetric + real diagonal)
    eigvals_qb, eigvecs_qb = np.linalg.eigh(H_eff.real)
    phases = np.exp(-1j * eigvals_qb * T_FIXED)

    def run(seed_nodes, _n=N):
        valid = [idx_pro[s] for s in seed_nodes if s in idx_pro]
        if not valid: return np.zeros(_n)
        psi0 = np.zeros(N_PRO, dtype=complex)
        psi0[valid] = 1.0 / np.sqrt(len(valid))
        psi_t = eigvecs_qb @ (phases * (eigvecs_qb.conj().T @ psi0))
        sc = np.zeros(_n)
        sc[_pro_dst] = (np.abs(psi_t)**2)[_pro_src]
        return sc
    return run

# ── Tạo 3 option potentials ───────────────────────────────────────
# Option 1: Uniform V0 = graph_mean_deg ≈ 22 (để so sánh fair với NH)
potential_opt1 = {i: graph_mean_deg for i in cof_info}

# Option 2: Degree-proportional v_i = k_i
potential_opt2 = {i: float(k) for i, k in cof_info.items()}

# Option 3: Normalized-degree v_i = mean_deg * (k_i / mean_cof_deg)
potential_opt3 = {i: graph_mean_deg * (float(k) / cof_mean_deg)
                  for i, k in cof_info.items()}

print(f'\nPotential ranges:')
print(f'  Option 1 (uniform):     V = {graph_mean_deg:.1f} (all cofactors)')
v2 = list(potential_opt2.values())
print(f'  Option 2 (degree):      V ∈ [{min(v2):.0f}, {max(v2):.0f}], mean={np.mean(v2):.1f}')
v3 = list(potential_opt3.values())
print(f'  Option 3 (normalized):  V ∈ [{min(v3):.1f}, {max(v3):.1f}], mean={np.mean(v3):.1f}')

# ── Build runners ─────────────────────────────────────────────────
print('\nBuilding methods...')

# CTQW-PRO baseline (standard, no modification)
phases_std = np.exp(-1j * eigvals_pro * T_FIXED)

def run_ctqw(seeds, _n=N):
    valid = [idx_pro[s] for s in seeds if s in idx_pro]
    if not valid: return np.zeros(_n)
    psi0 = np.zeros(N_PRO, dtype=complex)
    psi0[valid] = 1.0/np.sqrt(len(valid))
    psi_t = eigvecs_pro @ (phases_std * (eigvecs_pro.conj().T @ psi0))
    sc = np.zeros(_n); sc[_pro_dst] = (np.abs(psi_t)**2)[_pro_src]
    return sc

# NH-CTQW-PRO (imaginary decay baseline để so sánh trực tiếp)
print('  Building NH-CTQW-PRO...')
run_nh = make_nh_pro(
    A_pro, idx_pro, N, N_PRO, _pro_src, _pro_dst,
    RECON3D_COFACTORS, pro_nodes, NH_GAMMA, T_FIXED)

# 3 quantum barrier options
print('  Building QB Option 1 (uniform)...')
run_qb1 = make_quantum_barrier(potential_opt1, 'QB-uniform')

print('  Building QB Option 2 (degree)...')
run_qb2 = make_quantum_barrier(potential_opt2, 'QB-degree')

print('  Building QB Option 3 (normalized)...')
run_qb3 = make_quantum_barrier(potential_opt3, 'QB-normalized')

print('  All built.')

# ── Sample diseases ───────────────────────────────────────────────
all_d = sorted(eval_set3.items(), key=lambda x: -len(x[1]))
if N_DISEASES and N_DISEASES < len(all_d):
    step = max(1, len(all_d) // N_DISEASES)
    sample = dict(list(all_d)[i] for i in range(0, len(all_d), step))
    sample = dict(list(sample.items())[:N_DISEASES])
else:
    sample = dict(all_d)
print(f'\nSample: {len(sample)} diseases\n')

# ── LOO evaluation ────────────────────────────────────────────────
def loo_eval(fn, dset, label=''):
    rows = []
    for disease, mets in tqdm(dset.items(), desc=label, leave=False):
        valid = [m for m in mets if m in node_idx]
        if len(valid) < 3: continue
        fold_res = []
        for i, test_met in enumerate(valid):
            seeds    = [m for j, m in enumerate(valid) if j != i]
            seed_set = {node_idx[s] for s in seeds}
            try:
                sc = fn(seeds)
                m  = compute_metrics(sc, node_idx[test_met], seed_set, _n=N)
                if m: fold_res.append(m)
            except Exception:
                pass
        if fold_res:
            row = {k: float(np.mean([r[k] for r in fold_res]))
                   for k in ['auc','mrr','r@5','r@20']}
            row['disease'] = disease
            rows.append(row)
    return pd.DataFrame(rows) if rows else None

# ── Run ───────────────────────────────────────────────────────────
methods = [
    ('CTQW-PRO (baseline)',  run_ctqw),
    ('NH-CTQW-PRO (γ=22)',   run_nh),
    ('QB-1 uniform (V=22)',  run_qb1),
    ('QB-2 degree (V=k_i)',  run_qb2),
    ('QB-3 norm (V∝k/k̄)',   run_qb3),
]

print(f"{'Method':<28} {'AUC':>8} {'MRR':>8} {'R@5':>7} {'R@20':>7}")
print('─'*60)

results = {}
for name, fn in methods:
    df = loo_eval(fn, sample, label=name)
    if df is not None and not df.empty:
        results[name] = df
        print(f"  {name:<26} "
              f"{df['auc'].mean():>8.4f} "
              f"{df['mrr'].mean():>8.4f} "
              f"{df['r@5'].mean():>7.4f} "
              f"{df['r@20'].mean():>7.4f}")

# ── Delta vs CTQW-PRO baseline ────────────────────────────────────
print(f"\nΔ vs CTQW-PRO baseline:")
print(f"{'Method':<28} {'ΔMRR':>10} {'ΔR@20':>10}  Verdict")
print('─'*60)
base_key = 'CTQW-PRO (baseline)'
if base_key in results:
    base = results[base_key]
    for name, _ in methods[1:]:
        if name not in results: continue
        shared = sorted(set(base['disease']) & set(results[name]['disease']))
        if not shared: continue
        b = base.set_index('disease').loc[shared]
        c = results[name].set_index('disease').loc[shared]
        dm  = (c['mrr']  - b['mrr']).mean()
        dr  = (c['r@20'] - b['r@20']).mean()
        v   = ('✅ Better'   if dm >  0.005 else
               '❌ Worse'    if dm < -0.005 else '≈  No change')
        print(f"  {name:<26} {dm:>+10.4f} {dr:>+10.4f}  {v}")

# ── Insight: degree correlation ───────────────────────────────────
print(f'\n── Degree correlation (hub bias test, 10 diseases) ──')
from scipy import stats as sp_stats

n_test = min(10, len(list(sample.items())))
test_sample = dict(list(sample.items())[:n_test])

for name, fn in methods:
    rwr_scores_all = []; ctqw_scores_all = []; deg_all = []
    gcc_to_pro = {node_idx[nd]: idx_pro[nd]
                  for nd in node_idx if nd in idx_pro}
    # cofactor gcc idx
    cof_gcc = {node_idx[nd] for nd in node_idx
               if nd in cof_set or
               nd.replace('_c','').replace('_m','').replace('_e','') in cof_set}

    for disease, mets in test_sample.items():
        valid = [m for m in mets if m in node_idx]
        if len(valid) < 3: continue
        sc = fn(valid)
        for i in range(N):
            if i in {node_idx[s] for s in valid}: continue
            if i in cof_gcc: continue
            if i not in gcc_to_pro: continue
            ctqw_scores_all.append(sc[i])
            deg_all.append(deg_pro[gcc_to_pro[i]])

    if len(deg_all) > 10:
        r, _ = sp_stats.pearsonr(deg_all, ctqw_scores_all)
        print(f"  {name:<28} r={r:+.3f}")

print('\nDone.')
print("""
INTERPRETATION:
  r gần 0  → ít hub bias (tốt)
  r > 0    → còn hub bias (nhều → tệ hơn)

  QB methods dùng real potential → phase detuning, không phải amplitude decay
  So sánh với NH (imaginary) để thấy cơ chế nào hiệu quả hơn.
""")

Setup...
  Extracting SMPDB metabolites...


Extract SMPDB:   0%|          | 0/48687 [00:00<?, ?it/s]

  Disease pathways: 20248


SMPDB files:   0%|          | 0/48687 [00:00<?, ?it/s]

  Loaded 20244 pathways (errors=0)
  Dedup: 20244 → 3240 → 362 → 215
  SMPDB: 153 diseases
  Cofactor nodes: 42
  Cofactor mean degree: 349.3
  Graph mean degree (NH_GAMMA): 21.7

Potential ranges:
  Option 1 (uniform):     V = 21.7 (all cofactors)
  Option 2 (degree):      V ∈ [2, 2457], mean=349.3
  Option 3 (normalized):  V ∈ [0.1, 152.5], mean=21.7

Building methods...
  Building NH-CTQW-PRO...
  Building QB Option 1 (uniform)...
  Building QB Option 2 (degree)...
  Building QB Option 3 (normalized)...
  All built.

Sample: 30 diseases

Method                            AUC      MRR     R@5    R@20
────────────────────────────────────────────────────────────


...

  QB-1 uniform (V=22)          0.9591   0.1774  0.2603  0.4909


QB-2 degree (V=k_i):   0%|          | 0/30 [00:00<?, ?it/s]

  QB-2 degree (V=k_i)          0.9621   0.2637  0.3573  0.5769


QB-3 norm (V∝k/k̄):   0%|          | 0/30 [00:00<?, ?it/s]

  QB-3 norm (V∝k/k̄)           0.9547   0.1835  0.2640  0.5051

Δ vs CTQW-PRO baseline:
Method                             ΔMRR      ΔR@20  Verdict
────────────────────────────────────────────────────────────
  NH-CTQW-PRO (γ=22)            +0.0475    +0.0423  ✅ Better
  QB-1 uniform (V=22)           -0.0269    -0.0384  ❌ Worse
  QB-2 degree (V=k_i)           +0.0594    +0.0475  ✅ Better
  QB-3 norm (V∝k/k̄)            -0.0208    -0.0242  ❌ Worse

── Degree correlation (hub bias test, 10 diseases) ──
  CTQW-PRO (baseline)          r=+0.349
  NH-CTQW-PRO (γ=22)           r=+0.305
  QB-1 uniform (V=22)          r=+0.274
  QB-2 degree (V=k_i)          r=+0.282
  QB-3 norm (V∝k/k̄)           r=+0.295

Done.

INTERPRETATION:
  r gần 0  → ít hub bias (tốt)
  r > 0    → còn hub bias (nhều → tệ hơn)
  
  QB methods dùng real potential → phase detuning, không phải amplitude decay
  So sánh với NH (imaginary) để thấy cơ chế nào hiệu quả hơn.



In [6]:
# ═══════════════════════════════════════════════════════════════════
# TEST: QB-2 (V=k_i) full evaluation trên cả 3 datasets
# So sánh: CTQW-PRO vs NH-CTQW-PRO vs QB-2 vs QB-2+RRF(Driven)
# Copy toàn bộ vào 1 cell Colab (~40-50 phút)
# ═══════════════════════════════════════════════════════════════════
import sys, os
for _p in ['/content/project/src',
           '/content/drive/MyDrive/CTQW for metabolites/src']:
    if os.path.isdir(_p) and _p not in sys.path:
        sys.path.insert(0, _p)

import numpy as np
import pandas as pd
from scipy import stats as sp_stats
from tqdm.auto import tqdm

from config import (CACHE_DIR, RESULTS_DIR, T_FIXED, NH_GAMMA,
                    RECON3D_COFACTORS, DRIVEN_N_STEPS, DRIVEN_ALPHA, RRF_K)
from graph import (parse_recon3d, build_gcc, build_gpro,
                   build_hmdb_to_recon_initial, augment_hmdb_to_recon,
                   compute_eigendecomp)
from eval_sets import (parse_hmdb, build_hmdb_lookups, build_cofactors_set,
                       build_eval_set1, build_eval_set2, build_eval_set3)
from methods import make_nh_pro, make_driven_pro, make_rrf
from evaluation import run_loo_eval, wilcoxon_table

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Setup ─────────────────────────────────────────────────────────
print('Setup...')
recon_data   = parse_recon3d()
G_cc, graph_nodes, N, node_idx, A_cc, degrees = build_gcc(recon_data)
met_info     = recon_data['met_info']
pathway_mets = recon_data['pathway_mets']

(G_pro, pro_nodes, N_PRO, idx_pro,
 A_pro, deg_pro, _pro_src, _pro_dst) = build_gpro(G_cc, node_idx, pathway_mets)

eigvals, eigvecs = compute_eigendecomp(A_pro, CACHE_DIR / 'gpro_eigdecomp.npz')

hmdb_data        = parse_hmdb()
hmdb_metabolites = hmdb_data['metabolites']
hmdb_lookups     = build_hmdb_lookups(hmdb_metabolites)
hmdb_to_recon    = build_hmdb_to_recon_initial(met_info, node_idx)
augment_hmdb_to_recon(
    hmdb_to_recon, met_info, node_idx,
    hmdb_lookups['ik_to_id'], hmdb_lookups['ikshort_to_id'],
    hmdb_lookups['name_to_id'], hmdb_lookups['name_aggr_to_id'])
COFACTORS = build_cofactors_set(hmdb_metabolites)

eval_set1, disease_canonical = build_eval_set1(
    hmdb_metabolites, hmdb_lookups, hmdb_to_recon, node_idx, COFACTORS)
eval_set2     = build_eval_set2(
    hmdb_metabolites, hmdb_lookups, hmdb_to_recon,
    node_idx, COFACTORS, disease_canonical)
eval_set3     = build_eval_set3(hmdb_metabolites, hmdb_to_recon, node_idx)
print(f'  HMDB+CTD: {len(eval_set1)}, MarkerDB: {len(eval_set2)}, SMPDB: {len(eval_set3)}')

# ── QB-2 builder ──────────────────────────────────────────────────
cof_set  = set(RECON3D_COFACTORS)
cof_info = {}
for i, nd in enumerate(pro_nodes):
    nd_base = nd.replace('_c','').replace('_m','').replace('_e','').replace('_x','')
    if nd in cof_set or nd_base in cof_set:
        cof_info[i] = deg_pro[i]
print(f'  Cofactor nodes: {len(cof_info)}, '
      f'mean degree: {np.mean(list(cof_info.values())):.1f}')

def make_qb2(A_pro, idx_pro, N, N_PRO, src, dst, cof_info, t=T_FIXED):
    """
    Quantum Barrier Option 2: H = A_pro + diag(k_i) at cofactor nodes
    Real on-site potential proportional to node degree.
    Hermitian → eigh, not eig.
    """
    H = A_pro.copy().astype(float)
    for i, k in cof_info.items():
        H[i, i] += float(k)          # real diagonal → Hermitian

    ev_qb, vecs_qb = np.linalg.eigh(H)   # symmetric → eigh
    phases = np.exp(-1j * ev_qb * t)

    def run(seed_nodes, _n=N):
        valid = [idx_pro[s] for s in seed_nodes if s in idx_pro]
        if not valid: return np.zeros(_n)
        psi0 = np.zeros(N_PRO, dtype=complex)
        psi0[valid] = 1.0 / np.sqrt(len(valid))
        psi_t = vecs_qb @ (phases * (vecs_qb.conj().T @ psi0))
        sc = np.zeros(_n)
        sc[dst] = (np.abs(psi_t)**2)[src]
        return sc
    return run

# ── Build all methods ─────────────────────────────────────────────
print('\nBuilding methods...')

# CTQW-PRO baseline
phases_std = np.exp(-1j * eigvals * T_FIXED)
def run_ctqw(seeds, _n=N):
    valid = [idx_pro[s] for s in seeds if s in idx_pro]
    if not valid: return np.zeros(_n)
    psi0 = np.zeros(N_PRO, dtype=complex)
    psi0[valid] = 1.0/np.sqrt(len(valid))
    psi_t = eigvecs @ (phases_std * (eigvecs.conj().T @ psi0))
    sc = np.zeros(_n); sc[_pro_dst] = (np.abs(psi_t)**2)[_pro_src]
    return sc

# NH-CTQW-PRO
print('  NH-CTQW-PRO...')
run_nh = make_nh_pro(
    A_pro, idx_pro, N, N_PRO, _pro_src, _pro_dst,
    RECON3D_COFACTORS, pro_nodes, NH_GAMMA, T_FIXED)

# QB-2
print('  QB-2 (V=k_i)...')
run_qb2 = make_qb2(
    A_pro, idx_pro, N, N_PRO, _pro_src, _pro_dst, cof_info)

# Driven (dùng eigvals/eigvecs của A_pro gốc — consistent với Table 4)
run_driven = make_driven_pro(
    eigvals, eigvecs, idx_pro, N, N_PRO, _pro_src, _pro_dst,
    n_steps=DRIVEN_N_STEPS, alpha=DRIVEN_ALPHA)

# RRF: NH + Driven (current paper)
run_rrf_nh = make_rrf(run_nh, run_driven, k=RRF_K)

# RRF: QB-2 + Driven (new variant)
run_rrf_qb2 = make_rrf(run_qb2, run_driven, k=RRF_K)

print('  All built.\n')

# ── Full LOO evaluation ───────────────────────────────────────────
datasets = [
    ('HMDB+CTD', eval_set1),
    ('MarkerDB',  eval_set2),
    ('SMPDB',     eval_set3),
]

methods = [
    ('CTQW-PRO',      run_ctqw),
    ('NH (γ=22)',      run_nh),
    ('QB-2 (V=k_i)',   run_qb2),
    ('Driven',         run_driven),
    ('RRF(NH+Driven)', run_rrf_nh),
    ('RRF(QB2+Driven)',run_rrf_qb2),
]

all_results = {}   # {(method_name, dataset): df}

for ds_label, dset in datasets:
    print(f'── {ds_label} ──')
    for m_name, fn in methods:
        key = (m_name, ds_label)
        df  = run_loo_eval(dset, fn, node_idx, N,
                           label=f'{m_name}/{ds_label}')
        all_results[key] = df

# ── Print summary tables ──────────────────────────────────────────
print('\n' + '='*72)
print('FULL RESULTS: CTQW-PRO vs NH vs QB-2')
print('='*72)

for ds_label, _ in datasets:
    print(f'\n=== {ds_label} ===')
    print(f"{'Method':<22} {'AUC':>8} {'MRR':>8} {'R@5':>7} {'R@10':>7} {'R@20':>7}")
    print('-'*60)
    for m_name, _ in methods:
        df = all_results.get((m_name, ds_label))
        if df is None or df.empty: continue
        print(f"  {m_name:<20} "
              f"{df['auc'].mean():>8.4f} "
              f"{df['mrr'].mean():>8.4f} "
              f"{df['r@5'].mean():>7.4f} "
              f"{df['r@10'].mean():>7.4f} "
              f"{df['r@20'].mean():>7.4f}")

# ── Wilcoxon: QB-2 vs NH, QB-2 vs CTQW-PRO ───────────────────────
print('\n' + '='*72)
print('WILCOXON: QB-2 vs NH and vs CTQW-PRO')
print('='*72)

for ds_label, _ in datasets:
    df_ctqw = all_results.get(('CTQW-PRO',    ds_label))
    df_nh   = all_results.get(('NH (γ=22)',    ds_label))
    df_qb2  = all_results.get(('QB-2 (V=k_i)',ds_label))
    if df_qb2 is None or df_ctqw is None: continue

    print(f'\n{ds_label}:')
    wilcoxon_table(df_qb2, df_ctqw, f'{ds_label}',
                   method_a='QB-2', method_b='CTQW-PRO')
    if df_nh is not None:
        wilcoxon_table(df_qb2, df_nh, f'{ds_label}',
                       method_a='QB-2', method_b='NH')

# ── Degree correlation: QB-2 vs NH ───────────────────────────────
print('\n' + '='*50)
print('DEGREE CORRELATION (hub bias) — SMPDB 20 diseases')
print('='*50)

dset_test = dict(list(eval_set3.items())[:20])
cof_gcc   = {node_idx[nd] for nd in node_idx
             if nd in cof_set or
             nd.replace('_c','').replace('_m','').replace('_e','') in cof_set}
gcc_to_pro = {node_idx[nd]: idx_pro[nd]
              for nd in node_idx if nd in idx_pro}

for m_name, fn in [('CTQW-PRO', run_ctqw),
                   ('NH (γ=22)', run_nh),
                   ('QB-2 (V=k_i)', run_qb2)]:
    sc_all = []; deg_all = []
    for disease, mets in dset_test.items():
        valid = [m for m in mets if m in node_idx]
        if len(valid) < 3: continue
        sc = fn(valid)
        seed_set = {node_idx[s] for s in valid}
        for i in range(N):
            if i in seed_set or i in cof_gcc: continue
            if i not in gcc_to_pro: continue
            sc_all.append(sc[i])
            deg_all.append(deg_pro[gcc_to_pro[i]])
    if len(deg_all) > 10:
        r, _ = sp_stats.pearsonr(deg_all, sc_all)
        print(f'  {m_name:<22} r = {r:+.3f}')

# ── Save ──────────────────────────────────────────────────────────
save_rows = []
for (m_name, ds_label), df in all_results.items():
    if df is not None and not df.empty:
        tmp = df.copy()
        tmp['method']  = m_name
        tmp['dataset'] = ds_label
        save_rows.append(tmp)
if save_rows:
    out = RESULTS_DIR / 'qb2_full_results.csv'
    pd.concat(save_rows).to_csv(out, index=False)
    print(f'\nSaved: {out}')
print('Done.')

Setup...
  Disease pathways: 20248


SMPDB files:   0%|          | 0/48687 [00:00<?, ?it/s]

  Loaded 20244 pathways (errors=0)
  Dedup: 20244 → 3240 → 362 → 215
  HMDB+CTD: 158, MarkerDB: 21, SMPDB: 153
  Cofactor nodes: 42, mean degree: 349.3

Building methods...
  NH-CTQW-PRO...
  QB-2 (V=k_i)...
  All built.

── HMDB+CTD ──


CTQW-PRO/HMDB+CTD:   0%|          | 0/158 [00:00<?, ?it/s]

NH (γ=22)/HMDB+CTD:   0%|          | 0/158 [00:00<?, ?it/s]

...

QB-2 (V=k_i)/SMPDB:   0%|          | 0/153 [00:00<?, ?it/s]

Driven/SMPDB:   0%|          | 0/153 [00:00<?, ?it/s]

RRF(NH+Driven)/SMPDB:   0%|          | 0/153 [00:00<?, ?it/s]

RRF(QB2+Driven)/SMPDB:   0%|          | 0/153 [00:00<?, ?it/s]


FULL RESULTS: CTQW-PRO vs NH vs QB-2

=== HMDB+CTD ===
Method                      AUC      MRR     R@5    R@10    R@20
------------------------------------------------------------
  CTQW-PRO               0.8195   0.0598  0.0848  0.1340  0.1915
  NH (γ=22)              0.8216   0.0658  0.0955  0.1450  0.2010
  QB-2 (V=k_i)           0.7882   0.0612  0.0919  0.1333  0.1831
  Driven                 0.7973   0.0594  0.0853  0.1459  0.1913
  RRF(NH+Driven)         0.8168   0.0686  0.0972  0.1480  0.2001
  RRF(QB2+Driven)        0.8003   0.0638  0.0915  0.1400  0.1910

=== MarkerDB ===
Method                      AUC      MRR     R@5    R@10    R@20
------------------------------------------------------------
  CTQW-PRO               0.8835   0.0850  0.1299  0.1858  0.2646
  NH (γ=22)              0.8877   0.0918  0.1260  0.1923  0.2622
  QB-2 (V=k_i)           0.8749   0.0828  0.1275  0.1892  0.2580
  Driven                 0.8749   0.0775  0.1106  0.1605  0.2225
  RRF(NH+Driven)        

In [10]:
# ═══════════════════════════════════════════════════════════════════
# TEST: QB-2 + RRF variants — full evaluation cả 3 datasets
# OPTIMIZED: tính NH, QB-2, Driven 1 lần per fold, tái dùng cho RRF
# Methods:
#   1. CTQW-PRO
#   2. NH (γ=22)
#   3. QB-2 (V=k_i)
#   4. RRF(NH+QB2)
#   5. RRF(NH+QB2+Driven)
# Copy toàn bộ vào 1 cell Colab (~50-60 phút)
# ═══════════════════════════════════════════════════════════════════
import sys, os
for _p in ['/content/project/src',
           '/content/drive/MyDrive/CTQW for metabolites/src']:
    if os.path.isdir(_p) and _p not in sys.path:
        sys.path.insert(0, _p)

import numpy as np
import pandas as pd
from scipy.stats import rankdata as _rankdata
from tqdm.auto import tqdm

from config import (CACHE_DIR, RESULTS_DIR, T_FIXED, NH_GAMMA,
                    RECON3D_COFACTORS, DRIVEN_N_STEPS, DRIVEN_ALPHA, RRF_K)
from graph import (parse_recon3d, build_gcc, build_gpro,
                   build_hmdb_to_recon_initial, augment_hmdb_to_recon,
                   compute_eigendecomp)
from eval_sets import (parse_hmdb, build_hmdb_lookups, build_cofactors_set,
                       build_eval_set1, build_eval_set2, build_eval_set3)
from methods import make_nh_pro, make_driven_pro
from evaluation import compute_metrics, wilcoxon_table

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Setup ─────────────────────────────────────────────────────────
print('Setup...')
recon_data   = parse_recon3d()
G_cc, graph_nodes, N, node_idx, A_cc, degrees = build_gcc(recon_data)
met_info     = recon_data['met_info']
pathway_mets = recon_data['pathway_mets']

(G_pro, pro_nodes, N_PRO, idx_pro,
 A_pro, deg_pro, _pro_src, _pro_dst) = build_gpro(G_cc, node_idx, pathway_mets)

eigvals, eigvecs = compute_eigendecomp(A_pro, CACHE_DIR / 'gpro_eigdecomp.npz')

hmdb_data        = parse_hmdb()
hmdb_metabolites = hmdb_data['metabolites']
hmdb_lookups     = build_hmdb_lookups(hmdb_metabolites)
hmdb_to_recon    = build_hmdb_to_recon_initial(met_info, node_idx)
augment_hmdb_to_recon(
    hmdb_to_recon, met_info, node_idx,
    hmdb_lookups['ik_to_id'], hmdb_lookups['ikshort_to_id'],
    hmdb_lookups['name_to_id'], hmdb_lookups['name_aggr_to_id'])
COFACTORS = build_cofactors_set(hmdb_metabolites)

eval_set1, disease_canonical = build_eval_set1(
    hmdb_metabolites, hmdb_lookups, hmdb_to_recon, node_idx, COFACTORS)
eval_set2 = build_eval_set2(
    hmdb_metabolites, hmdb_lookups, hmdb_to_recon,
    node_idx, COFACTORS, disease_canonical)
eval_set3 = build_eval_set3(hmdb_metabolites, hmdb_to_recon, node_idx)
print(f'  HMDB+CTD: {len(eval_set1)}, MarkerDB: {len(eval_set2)}, '
      f'SMPDB: {len(eval_set3)}')

# ── Cofactor info ─────────────────────────────────────────────────
cof_set  = set(RECON3D_COFACTORS)
cof_info = {
    i: deg_pro[i] for i, nd in enumerate(pro_nodes)
    if nd in cof_set or
    nd.replace('_c','').replace('_m','').replace('_e','').replace('_x','') in cof_set
}
print(f'  Cofactor nodes: {len(cof_info)}, '
      f'mean degree: {np.mean(list(cof_info.values())):.1f}')

# ── Build primitive runners ───────────────────────────────────────
# CTQW-PRO
phases_std = np.exp(-1j * eigvals * T_FIXED)
def run_ctqw(seeds, _n=N):
    valid = [idx_pro[s] for s in seeds if s in idx_pro]
    if not valid: return np.zeros(_n)
    psi0 = np.zeros(N_PRO, dtype=complex)
    psi0[valid] = 1.0 / np.sqrt(len(valid))
    psi_t = eigvecs @ (phases_std * (eigvecs.conj().T @ psi0))
    sc = np.zeros(_n); sc[_pro_dst] = (np.abs(psi_t)**2)[_pro_src]
    return sc

print('\nBuilding methods...')

# QB-2: H = A_pro + diag(k_i) — real, Hermitian
print('  QB-2 (V=k_i)...')
def make_qb2(t=T_FIXED):
    H = A_pro.copy().astype(float)
    for i, k in cof_info.items():
        H[i, i] += float(k)
    ev, vecs = np.linalg.eigh(H)
    ph = np.exp(-1j * ev * t)
    def run(seeds, _n=N):
        valid = [idx_pro[s] for s in seeds if s in idx_pro]
        if not valid: return np.zeros(_n)
        psi0 = np.zeros(N_PRO, dtype=complex)
        psi0[valid] = 1.0 / np.sqrt(len(valid))
        psi_t = vecs @ (ph * (vecs.conj().T @ psi0))
        sc = np.zeros(_n); sc[_pro_dst] = (np.abs(psi_t)**2)[_pro_src]
        return sc
    return run
run_qb2 = make_qb2()

print('  NH-CTQW-PRO...')
run_nh = make_nh_pro(
    A_pro, idx_pro, N, N_PRO, _pro_src, _pro_dst,
    RECON3D_COFACTORS, pro_nodes, NH_GAMMA, T_FIXED)

print('  Driven...')
run_driven = make_driven_pro(
    eigvals, eigvecs, idx_pro, N, N_PRO, _pro_src, _pro_dst,
    n_steps=DRIVEN_N_STEPS, alpha=DRIVEN_ALPHA)

print('  All built.\n')

# ── RRF helpers ───────────────────────────────────────────────────
def rrf2(sc_a, sc_b, k=RRF_K):
    """RRF từ 2 score vectors đã tính sẵn."""
    ra = _rankdata(-sc_a, method='average')
    rb = _rankdata(-sc_b, method='average')
    return 1.0/(k + ra) + 1.0/(k + rb)

def rrf3(sc_a, sc_b, sc_c, k=RRF_K):
    """RRF từ 3 score vectors đã tính sẵn — equal weight."""
    ra = _rankdata(-sc_a, method='average')
    rb = _rankdata(-sc_b, method='average')
    rc = _rankdata(-sc_c, method='average')
    return 1.0/(k + ra) + 1.0/(k + rb) + 1.0/(k + rc)

# ── Custom LOO: tính tất cả methods trong 1 fold pass ─────────────
METHOD_NAMES = [
    'CTQW-PRO',
    'NH (γ=22)',
    'QB-2 (V=k_i)',
    'RRF(NH+QB2)',
    'RRF(NH+QB2+Driven)',
]

def loo_all_methods(disease_set, label=''):
    """
    Single-pass LOO: tính NH, QB-2, Driven 1 lần per fold,
    tái dùng ngay cho RRF — không tính lại.
    """
    # Accumulators: {method_name: [per-disease mean dict]}
    acc = {m: [] for m in METHOD_NAMES}

    for disease, mets in tqdm(disease_set.items(),
                               desc=label or 'LOO', leave=True):
        valid = [m for m in mets if m in node_idx]
        if len(valid) < 3: continue

        # Per-fold results: {method_name: [metric_dict]}
        fold_acc = {m: [] for m in METHOD_NAMES}

        for i, test_met in enumerate(valid):
            seeds    = [m for j, m in enumerate(valid) if j != i]
            seed_set = {node_idx[s] for s in seeds}
            test_idx = node_idx[test_met]

            # ── Compute primitives ONCE ────────────────────────────
            try:
                sc_ctqw   = run_ctqw(seeds)
                sc_nh     = run_nh(seeds)
                sc_qb2    = run_qb2(seeds)
                sc_driven = run_driven(seeds)
            except Exception:
                continue

            # ── Derive RRF scores from cached primitives ───────────
            sc_rrf2 = rrf2(sc_nh, sc_qb2)
            sc_rrf3 = rrf3(sc_nh, sc_qb2, sc_driven)

            # ── Compute metrics for all methods ────────────────────
            score_map = {
                'CTQW-PRO':            sc_ctqw,
                'NH (γ=22)':           sc_nh,
                'QB-2 (V=k_i)':        sc_qb2,
                'RRF(NH+QB2)':         sc_rrf2,
                'RRF(NH+QB2+Driven)':  sc_rrf3,
            }
            for m_name, sc in score_map.items():
                met = compute_metrics(sc, test_idx, seed_set, _n=N)
                if met:
                    fold_acc[m_name].append(met)

        # Per-disease mean
        for m_name in METHOD_NAMES:
            folds = fold_acc[m_name]
            if folds:
                row = {k: float(np.mean([r[k] for r in folds]))
                       for k in ['auc','mrr','r@5','r@10','r@20','r@50']}
                row['disease'] = disease
                acc[m_name].append(row)

    return {m: pd.DataFrame(rows) if rows else None
            for m, rows in acc.items()}

# ── Run evaluation ────────────────────────────────────────────────
datasets = [
    ('HMDB+CTD', eval_set1),
    ('MarkerDB',  eval_set2),
    ('SMPDB',     eval_set3),
]

all_results = {}  # {(method_name, dataset): df}
for ds_label, dset in datasets:
    print(f'── {ds_label} ──')
    res = loo_all_methods(dset, label=ds_label)
    for m_name, df in res.items():
        all_results[(m_name, ds_label)] = df

# ── Print summary ─────────────────────────────────────────────────
print('\n' + '='*72)
print('RESULTS: QB-2 + RRF variants (optimized single-pass LOO)')
print('='*72)

# Reference: previous RRF(NH+Driven) từ qb2_full_results.csv
# (main_results.csv dùng tên 'RRF' không match — chỉ load từ qb2 file)
prev = {}
fpath = RESULTS_DIR / 'qb2_full_results.csv'
if fpath.exists():
    df_p = pd.read_csv(fpath)
    for ds in ['HMDB+CTD', 'MarkerDB', 'SMPDB']:
        sub = df_p[(df_p['method']=='RRF(NH+Driven)') & (df_p['dataset']==ds)]
        if not sub.empty:
            prev[('RRF(NH+Driven)', ds)] = sub

for ds_label, _ in datasets:
    print(f'\n=== {ds_label} ===')
    print(f"{'Method':<26} {'AUC':>8} {'MRR':>8} {'R@5':>7} "
          f"{'R@10':>7} {'R@20':>7} {'R@50':>7}")
    print('-'*72)
    for m_name in METHOD_NAMES:
        df = all_results.get((m_name, ds_label))
        if df is None or df.empty: continue
        r50 = df['r@50'].mean() if 'r@50' in df.columns else float('nan')
        print(f"  {m_name:<24} "
              f"{df['auc'].mean():>8.4f} "
              f"{df['mrr'].mean():>8.4f} "
              f"{df['r@5'].mean():>7.4f} "
              f"{df['r@10'].mean():>7.4f} "
              f"{df['r@20'].mean():>7.4f} "
              f"{r50:>7.4f}")
    # Reference
    for ref in ['RRF(NH+Driven)']:
        df = prev.get((ref, ds_label))
        if df is None or df.empty: continue
        r50 = df['r@50'].mean() if 'r@50' in df.columns else float('nan')
        print(f"  {ref:<24} "
              f"{df['auc'].mean():>8.4f} "
              f"{df['mrr'].mean():>8.4f} "
              f"{df['r@5'].mean():>7.4f} "
              f"{df['r@10'].mean():>7.4f} "
              f"{df['r@20'].mean():>7.4f} "
              f"{r50:>7.4f}  ◄ prev")

# ── Wilcoxon ─────────────────────────────────────────────────────
print('\n' + '='*72)
print('WILCOXON: key comparisons')
print('='*72)

comparisons = [
    ('RRF(NH+QB2)',        'NH (γ=22)',           'vs NH'),
    ('RRF(NH+QB2)',        'QB-2 (V=k_i)',         'vs QB-2'),
    ('RRF(NH+QB2+Driven)', 'RRF(NH+QB2)',          'vs RRF(NH+QB2)'),
    ('RRF(NH+QB2+Driven)', 'NH (γ=22)',            'vs NH'),
]

for ds_label, _ in datasets:
    print(f'\n── {ds_label} ──')
    for m_a, m_b, label in comparisons:
        df_a = all_results.get((m_a, ds_label))
        df_b = all_results.get((m_b, ds_label))
        if df_a is None or df_b is None: continue
        wilcoxon_table(df_a, df_b, f'{ds_label} {label}',
                       method_a=m_a, method_b=m_b)

# ── Save ──────────────────────────────────────────────────────────
save_rows = []
for (m_name, ds_label), df in all_results.items():
    if df is not None and not df.empty:
        tmp = df.copy()
        tmp['method'] = m_name; tmp['dataset'] = ds_label
        save_rows.append(tmp)
if save_rows:
    out = RESULTS_DIR / 'rrf_qb2_nh_results.csv'
    pd.concat(save_rows).to_csv(out, index=False)
    print(f'\nSaved: {out}')
print('Done.')

Setup...
  Disease pathways: 20248


SMPDB files:   0%|          | 0/48687 [00:00<?, ?it/s]

  Loaded 20244 pathways (errors=0)
  Dedup: 20244 → 3240 → 362 → 215
  HMDB+CTD: 158, MarkerDB: 21, SMPDB: 153
  Cofactor nodes: 42, mean degree: 349.3

Building methods...
  QB-2 (V=k_i)...
  NH-CTQW-PRO...
  Driven...
  All built.

── HMDB+CTD ──


HMDB+CTD:   0%|          | 0/158 [00:00<?, ?it/s]

── MarkerDB ──


MarkerDB:   0%|          | 0/21 [00:00<?, ?it/s]

── SMPDB ──


SMPDB:   0%|          | 0/153 [00:00<?, ?it/s]


RESULTS: QB-2 + RRF variants (optimized single-pass LOO)

=== HMDB+CTD ===
Method                          AUC      MRR     R@5    R@10    R@20    R@50
------------------------------------------------------------------------
  CTQW-PRO                   0.8195   0.0598  0.0848  0.1340  0.1915  0.2701
  NH (γ=22)                  0.8216   0.0658  0.0955  0.1450  0.2010  0.2707
  QB-2 (V=k_i)               0.7882   0.0612  0.0919  0.1333  0.1831  0.2556
  RRF(NH+QB2)                0.8130   0.0641  0.0936  0.1393  0.1941  0.2692
  RRF(NH+QB2+Driven)         0.8128   0.0663  0.0985  0.1427  0.1961  0.2728
  RRF(NH+Driven)             0.8168   0.0686  0.0972  0.1480  0.2001  0.2761  ◄ prev

=== MarkerDB ===
Method                          AUC      MRR     R@5    R@10    R@20    R@50
------------------------------------------------------------------------
  CTQW-PRO                   0.8835   0.0850  0.1299  0.1858  0.2646  0.3634
  NH (γ=22)                  0.8877   0.0918  0.1260  0.192

In [12]:
import sys, os
for _p in ['/content/project/src',
           '/content/drive/MyDrive/CTQW for metabolites/src']:
    if os.path.isdir(_p) and _p not in sys.path:
        sys.path.insert(0, _p)

import pandas as pd
import numpy as np
from scipy import stats
from config import RESULTS_DIR

df = pd.read_csv(RESULTS_DIR / 'dispersion_analysis.csv')
print("Columns:", df.columns.tolist())
print(f"Shape: {df.shape}")
print(df.head(3))

for dataset in ['HMDB+CTD', 'SMPDB']:
    if 'dataset' in df.columns:
        sub = df[df['dataset'] == dataset].copy()
    else:
        sub = df.copy()

    if 'dispersion' not in sub.columns or 'delta_mrr' not in sub.columns:
        print(f"\n{dataset}: columns missing — check CSV structure")
        continue

    p25 = sub['dispersion'].quantile(0.25)
    p75 = sub['dispersion'].quantile(0.75)

    conc = sub[sub['dispersion'] <= p25]['delta_mrr'].dropna().values
    disp = sub[sub['dispersion'] >= p75]['delta_mrr'].dropna().values

    # Mann-Whitney: H1 = concentrated ΔMRR > dispersed ΔMRR
    stat, pval = stats.mannwhitneyu(conc, disp, alternative='greater')

    print(f"\n{dataset} (n_conc={len(conc)}, n_disp={len(disp)}):")
    print(f"  Concentrated: mean ΔMRR = {conc.mean():.4f}")
    print(f"  Dispersed:    mean ΔMRR = {disp.mean():.4f}")
    print(f"  Mann-Whitney U (one-sided, conc > disp): p = {pval:.4e}")
    if pval < 0.001:
        print(f"  → ✅ p < 0.001 — gap significant")
    elif pval < 0.05:
        print(f"  → ⚠️  p = {pval:.3f} — significant but weak")
    else:
        print(f"  → ❌ p = {pval:.3f} — not significant")

Columns: ['source', 'group', 'n', 'delta_mrr', 'delta_r20', 'profancy_mrr', 'ctqw_mrr']
Shape: (4, 7)
     source         group   n  delta_mrr  delta_r20  profancy_mrr  ctqw_mrr
0  HMDB+CTD  CONCENTRATED  40   0.093736   0.210317      0.015482  0.109218
1  HMDB+CTD     DISPERSED  40   0.035407   0.089172      0.010555  0.045963
2     SMPDB  CONCENTRATED  40   0.197031   0.389907      0.028119  0.225150

HMDB+CTD: columns missing — check CSV structure

SMPDB: columns missing — check CSV structure


In [13]:
import sys, os
for _p in ['/content/project/src',
           '/content/drive/MyDrive/CTQW for metabolites/src']:
    if os.path.isdir(_p) and _p not in sys.path:
        sys.path.insert(0, _p)

import pandas as pd
import numpy as np
from scipy import stats
from config import RESULTS_DIR

# Load per-disease data từ qb2_full_results.csv hoặc main_results.csv
# Cần CTQW-PRO và PROFANCY per-disease MRR để tính delta per disease

for fname in ['qb2_full_results.csv', 'main_results.csv']:
    fpath = RESULTS_DIR / fname
    if fpath.exists():
        df = pd.read_csv(fpath)
        print(f"{fname}: {df.shape}")
        print(f"  methods: {df['method'].unique().tolist()}")
        print(f"  datasets: {df['dataset'].unique().tolist()}")
        break

# Cần: CTQW-PRO và PROFANCY per-disease MRR trên HMDB+CTD và SMPDB
# Tính dispersion per disease từ dispersion_analysis.csv structure

# Vì dispersion_analysis.csv chỉ có summary (4 rows),
# cần tính Jaccard dispersion per disease rồi merge với delta_mrr per disease

print("\nFile structure:")
print(df.head(3).to_string())

qb2_full_results.csv: (1992, 11)
  methods: ['CTQW-PRO', 'NH (γ=22)', 'QB-2 (V=k_i)', 'Driven', 'RRF(NH+Driven)', 'RRF(QB2+Driven)']
  datasets: ['HMDB+CTD', 'MarkerDB', 'SMPDB']

File structure:
        auc       mrr        rank       r@5      r@10      r@20      r@50                   disease  n_mets    method   dataset
0  0.895119  0.093652  278.514085  0.183099  0.197183  0.267606  0.359155  Eosinophilic esophagitis     142  CTQW-PRO  HMDB+CTD
1  0.877487  0.042379  315.000000  0.097778  0.128889  0.142222  0.262222         Colorectal cancer     225  CTQW-PRO  HMDB+CTD
2  0.864467  0.119668  367.888889  0.246914  0.246914  0.283951  0.395062             Schizophrenia      81  CTQW-PRO  HMDB+CTD


In [17]:
import pandas as pd
import numpy as np
from scipy import stats
from config import RESULTS_DIR

# Load per-disease data
df_main = pd.read_csv(RESULTS_DIR / 'main_results.csv')

# Lấy CTQW-PRO (t=0.1) và PROFANCY per-disease MRR
ctqw = df_main[df_main['method']=='t=0.1'][['disease','source','mrr']].rename(columns={'mrr':'mrr_ctqw'})
prof = df_main[df_main['method']=='PROFANCY'][['disease','source','mrr']].rename(columns={'mrr':'mrr_prof'})

merged = ctqw.merge(prof, on=['disease','source'])
merged['delta_mrr'] = merged['mrr_ctqw'] - merged['mrr_prof']

# Tính Jaccard dispersion per disease
# Load từ dispersion_analysis.csv để lấy P25/P75 cutoffs
disp_summary = pd.read_csv(RESULTS_DIR / 'dispersion_analysis.csv')
print("dispersion_analysis columns:", disp_summary.columns.tolist())
print(disp_summary)

# Vì không có per-disease dispersion trong file,
# dùng approach khác: chia theo delta_mrr quantile không đúng
# Cần per-disease Jaccard → phải tính lại

# ALTERNATIVE: dùng source để proxy dispersion
# SMPDB = concentrated (pathway-based), HMDB+CTD = dispersed (clinical)
# Đây là approach valid: compare ΔMRR distribution giữa 2 datasets

for src in ['HMDB+CTD', 'SMPDB']:
    sub = merged[merged['source']==src]['delta_mrr'].dropna()
    print(f"\n{src} (n={len(sub)}):")
    print(f"  mean ΔMRR = {sub.mean():.4f}")
    print(f"  median ΔMRR = {sub.median():.4f}")

# Mann-Whitney: SMPDB delta > HMDB+CTD delta
smpdb = merged[merged['source']=='SMPDB']['delta_mrr'].dropna()
hmdb  = merged[merged['source']=='HMDB+CTD']['delta_mrr'].dropna()
stat, pval = stats.mannwhitneyu(smpdb, hmdb, alternative='greater')
print(f"\nMann-Whitney (SMPDB delta > HMDB+CTD delta): p = {pval:.4e}")
print(f"→ {'✅ p<0.001' if pval<0.001 else 'p='+str(round(pval,4))}")

# Trong-dataset test: concentrated vs dispersed
# Cần per-disease Jaccard — tính từ saved data nếu có
disp_per_disease_file = RESULTS_DIR / 'eigenmode_overlap.csv'
if disp_per_disease_file.exists():
    df_disp = pd.read_csv(disp_per_disease_file)
    print("\neigenmode_overlap.csv columns:", df_disp.columns.tolist())
    # Merge với delta_mrr
    df_disp2 = df_disp.merge(merged, left_on=['disease','dataset'],
                              right_on=['disease','source'], how='inner')
    print(f"Merged: {len(df_disp2)} rows")

    for dataset in ['HMDB+CTD', 'SMPDB']:
        sub = df_disp2[df_disp2['dataset']==dataset].copy()
        if len(sub) < 10: continue
        p25 = sub['dispersion'].quantile(0.25)
        p75 = sub['dispersion'].quantile(0.75)
        conc = sub[sub['dispersion']<=p25]['delta_mrr'].values
        disp = sub[sub['dispersion']>=p75]['delta_mrr'].values
        stat, pval = stats.mannwhitneyu(conc, disp, alternative='greater')
        print(f"\n{dataset}: conc mean={conc.mean():.4f}, disp mean={disp.mean():.4f}")
        print(f"  Mann-Whitney p = {pval:.4e} {'✅' if pval<0.001 else ''}")

dispersion_analysis columns: ['source', 'group', 'n', 'delta_mrr', 'delta_r20', 'profancy_mrr', 'ctqw_mrr']
     source         group   n  delta_mrr  delta_r20  profancy_mrr  ctqw_mrr
0  HMDB+CTD  CONCENTRATED  40   0.093736   0.210317      0.015482  0.109218
1  HMDB+CTD     DISPERSED  40   0.035407   0.089172      0.010555  0.045963
2     SMPDB  CONCENTRATED  40   0.197031   0.389907      0.028119  0.225150
3     SMPDB     DISPERSED  29   0.146801   0.276993      0.028481  0.175283

HMDB+CTD (n=158):
  mean ΔMRR = 0.0485
  median ΔMRR = 0.0133

SMPDB (n=153):
  mean ΔMRR = 0.1898
  median ΔMRR = 0.1820

Mann-Whitney (SMPDB delta > HMDB+CTD delta): p = 1.4169e-31
→ ✅ p<0.001
